In [1]:
#----Graph Alignment and Entity Resolution--What if two hospitals or databases use different names for the same drug or biomarker?
"""
For example:
- Osimertinib
- TAGRISSO
- AZD9291
These names may refer to the same active drug, but the system should verify this using a trusted terminology source rather 
than blindly merging similar strings. Graph alignment extends this idea beyond individual names: it identifies 
corresponding entities and relationships across different graphs.

How to understand it---
1. Normalize names by ignoring capitalization and punctuation.
2. Check for an exact match in a known alias dictionary.
3. If there is no exact match, calculate a string-similarity score.
4. Suggest a candidate for review or leave the entity unresolved.

For instance, AZD-9291 becomes azd9291, allowing an exact match with the listed alias.
Fuzzy matching is not proof of clinical equivalence. Similar names can refer to different drugs, formulations, strengths, 
or combinations. The example's 0.80 threshold is arbitrary and must not be used as a clinical validation threshold.
"""
from difflib import SequenceMatcher
# Names appearing in an incoming hospital record.
incoming_names = [
    "Osimertinib",
    "TAGRISSO",
    "AZD-9291",
    "Gefitinib"
]
# A simplified reference dictionary. # Aliases are illustrative and should be verified against # an authoritative drug terminology source in production.
reference_drugs = {
    "Osimertinib": [
        "Osimertinib",
        "TAGRISSO",
        "AZD9291"
    ],
    "Gefitinib": [
        "Gefitinib",
        "IRESSA"
    ]
}
def normalize(text):
    return "".join(
        character.lower()
        for character in text
        if character.isalnum()
    )
def resolve_entity(name):
    normalized_name = normalize(name)
    for canonical, aliases in reference_drugs.items():                # Exact normalized alias match first.
        normalized_aliases = [normalize(alias) for alias in aliases]  
        if normalized_name in normalized_aliases:
            return canonical, 1.0, "Exact alias match"
    best_canonical = None                                  # If no exact match exists, find a fuzzy candidate
    best_score = 0.0
    for canonical, aliases in reference_drugs.items():
        for alias in aliases:
            score = SequenceMatcher(None, normalized_name, normalize(alias)).ratio()
            if score > best_score:
                best_score = score
                best_canonical = canonical
    if best_score >= 0.80:
        return best_canonical, best_score, "Review suggested"
    return None, best_score, "Unresolved"
for name in incoming_names:
    canonical, score, status = resolve_entity(name)
    print(
        f"{name:12} -> {canonical or 'Unknown'}"
        f" | score={score:.2f} | {status}"
    )

Osimertinib  -> Osimertinib | score=1.00 | Exact alias match
TAGRISSO     -> Osimertinib | score=1.00 | Exact alias match
AZD-9291     -> Osimertinib | score=1.00 | Exact alias match
Gefitinib    -> Gefitinib | score=1.00 | Exact alias match
